In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import plotly.express as px
import numpy as np

from utils import get_data

seed = 42
wine_X_tr, wine_X_val, wine_X_test, wine_y_tr, wine_y_val, wine_y_test = get_data(seed)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.discriminant_analysis import unique_labels


class NN(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dims=[16, 32, 16], dropout=0.0, epochs=200, batch_size=32, **kwargs):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.kwargs = kwargs
        
    def fit(self, X, y):
        torch.manual_seed(seed)
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.num_features_ = X.shape[1]
        
        label_to_idx = {label: idx for idx, label in enumerate(self.classes_)}
        print(label_to_idx)
        y_mapped = np.array([label_to_idx[label] for label in y])

        layer_dims = [self.num_features_, *self.hidden_dims, len(self.classes_)]
        layers = []
        for i in range(len(layer_dims)-2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            layers.append(nn.ReLU())
            if self.dropout:
                layers.append(nn.Dropout(self.dropout))
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))

        self.model_ = nn.Sequential(*layers)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model_.parameters(), **self.kwargs)

        self.model_.train()
        X_tensor = torch.as_tensor(X, dtype=torch.float32)
        y_tensor = torch.as_tensor(y_mapped, dtype=torch.long)

        for _ in range(self.epochs):
            for i in range(0, len(X), self.batch_size):
                batch_X = X_tensor[i : i + self.batch_size]
                batch_y = y_tensor[i : i + self.batch_size]
                optimizer.zero_grad()
                outputs = self.model_(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        X = check_array(X)

        self.model_.eval()
        with torch.inference_mode():
            X_tensor = torch.as_tensor(X, dtype=torch.float32)
            outputs = self.model_(X_tensor)
            probs = torch.softmax(outputs, dim=1)

        return probs.numpy()
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

# def plotly_plot(lines: list[tuple[list[tuple[Any, Any]], str]], x_name: str = "x axis", y_name: str = "y axis", label_name: str = "label", **kwargs):
#     df = pd.DataFrame([{x_name: x, y_name: y, label_name: label} for data, label in lines for x, y in data])
#     fig = px.line(df, x=x_name, y=y_name, color=label_name, **kwargs)
#     fig.show()

# def train_and_plot(params_list: list[dict], param_to_test: str, data_amount ):
#     train_accuracies = []
#     val_accuracies = []
#     for params in params_list:
#         for data_amount in 
#         model = NN(**params)
#         model.fit(wine_X_tr, wine_y_tr)
#         train_accuracy = model.score(wine_X_tr, wine_y_tr)
#         # train_accuracy, val_accuracy = train(epochs, **params)
#         # train_accuracies.append(train_accuracy)
#         # val_accuracies.append(val_accuracy)
#     plotly_plot(
#         list(zip(train_accuracies, [str(params[param_to_test]) for params in params_list])), "epoch", "training accuracy", param_to_test
#     )
#     plotly_plot(
#         list(zip(val_accuracies, [str(params[param_to_test]) for params in params_list])), "epoch", "validation accuracy", param_to_test
    # )

In [13]:
NN().fit(wine_X_tr, wine_y_tr)

{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6}


NN()

In [ ]:
from sklearn.model_selection import learning_curve


learning_curve(NN(), wine_X_tr, wine_y_tr)

{np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4, np.int64(6): 5}
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6}
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6}
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6}


In [50]:
train_and_plot(
    [
        {"hidden_dims": [16]},
        {"hidden_dims": [32]},
        {"hidden_dims": [64]},
        {"hidden_dims": [16, 32, 16]},
        {"hidden_dims": [16, 64, 16]},
        {"hidden_dims": [32, 64, 32]}
    ],
    "hidden_dims"
)

training with hidden_dims: [16], epochs: 300, and kwargs: {}
training with hidden_dims: [32], epochs: 300, and kwargs: {}
training with hidden_dims: [64], epochs: 300, and kwargs: {}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {}
training with hidden_dims: [16, 64, 16], epochs: 300, and kwargs: {}
training with hidden_dims: [32, 64, 32], epochs: 300, and kwargs: {}


In [51]:
train_and_plot(
    [
        {"lr": 0.0001},
        {"lr": 0.0005},
        {"lr": 0.001},
        {"lr": 0.005},
        {"lr": 0.01},
        {"lr": 0.05},
        {"lr": 0.1},
        {"lr": 0.5},
        {"lr": 1},
    ],
    "lr"
)

training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.0001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.0005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.01}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.05}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.1}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 0.5}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'lr': 1}


In [52]:
train_and_plot(
    [
        {"weight_decay": 0},
        {"weight_decay": 0.00001},
        {"weight_decay": 0.00005},
        {"weight_decay": 0.0001},
        {"weight_decay": 0.0005},
        {"weight_decay": 0.001},
        {"weight_decay": 0.005},
        {"weight_decay": 0.01},
        {"weight_decay": 0.05},
        {"weight_decay": 0.1},
    ],
    "weight_decay"
)

training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 1e-05}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 5e-05}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.0001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.0005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.001}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.005}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.01}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.05}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'weight_decay': 0.1}


In [53]:
train_and_plot(
    [
        {"dropout": 0},
        {"dropout": 0.1},
        {"dropout": 0.2},
        {"dropout": 0.3},
        {"dropout": 0.4},
        {"dropout": 0.5},
        {"dropout": 0.6},
        {"dropout": 0.7},
    ],
    "dropout"
)

training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.1}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.2}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.3}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.4}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.5}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.6}
training with hidden_dims: [16, 32, 16], epochs: 300, and kwargs: {'dropout': 0.7}
